# Mục 3b(iii) — Khảo sát ngoại lệ theo five number summary
**Võ Minh Huy — 2311110117**

Khảo sát 13 cột số gốc trên 299 hồ sơ sau N07, trước các phép làm mịn, giống SQL 06a. Không xóa hồ sơ, không sửa giá trị. MaHoSo và age_group là cột phụ, không tham gia khảo sát.

## 1. Đọc dữ liệu và tính phân vị
Nội suy tuyến tính tại vị trí (n − 1) × p, tương ứng PERCENTILE_CONT trong SQL. Dùng Decimal để giữ chính xác khi tính cận IQR; chỉ làm tròn bảng hiển thị.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/11_dien_ejection_fraction/ban_lam_viec_sau_ef_python.csv"
du_lieu = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
du_lieu["MaHoSo"] = du_lieu["MaHoSo"].astype(int)
cac_cot = [c for c in du_lieu.columns if c not in ["MaHoSo", "age_group"]]
for cot in cac_cot:
    du_lieu[cot] = du_lieu[cot].map(lambda x: Decimal(x) if x.strip() else None)

def phan_vi_linear(gia_tri, p):
    x = sorted(v for v in gia_tri if v is not None)
    if not x:
        return None
    vi_tri = Decimal(len(x) - 1) * Decimal(p)
    i = int(vi_tri)
    return x[i] + (vi_tri - i) * (x[min(i+1, len(x)-1)] - x[i])

def hien_thi_2_so(bang, cac_cot_so):
    ket_qua = bang.copy()
    for c in cac_cot_so:
        ket_qua[c] = ket_qua[c].map(lambda v: v.quantize(
            Decimal("0.01"), rounding=ROUND_HALF_UP) if v is not None else None)
    return ket_qua

pd.DataFrame({"SoHoSo":[len(du_lieu)], "SoCotKhaoSat":[len(cac_cot)]})

,SoHoSo,SoCotKhaoSat
0,299,13


## 2. Five number summary của 13 cột
Min, Q1, Median, Q3 và Max được tính từ giá trị không thiếu của từng cột.

In [2]:
dong_phan_vi = []
for cot in cac_cot:
    x = du_lieu[cot].dropna().tolist()
    dong_phan_vi.append({"TenCot":cot, "SoGiaTri":len(x),
        "Min":min(x), "Q1":phan_vi_linear(x, "0.25"),
        "Median":phan_vi_linear(x, "0.50"),
        "Q3":phan_vi_linear(x, "0.75"), "Max":max(x)})
five_number = pd.DataFrame(dong_phan_vi)
hien_thi_2_so(five_number, ["Min", "Q1", "Median", "Q3", "Max"])

,TenCot,SoGiaTri,Min,Q1,Median,Q3,Max
0,age,299,40.00,53.00,60.00,68.00,95.00
1,anaemia,299,0.00,0.00,0.00,1.00,1.00
2,creatinine_phosphokinase,299,23.00,120.00,249.00,582.00,7861.00
3,diabetes,299,0.00,0.00,0.00,1.00,1.00
4,ejection_fraction,299,14.00,30.00,38.00,45.00,80.00
5,high_blood_pressure,299,0.00,0.00,0.00,1.00,1.00
6,platelets,299,25100.00,217500.00,259763.43,299000.00,850000.00
7,serum_creatinine,299,0.50,1.00,1.10,1.40,9.40
8,serum_sodium,299,113.00,134.00,137.00,139.00,148.00
9,sex,299,0.00,0.00,1.00,1.00,1.00


## 3. Cận IQR và số ngoại lệ
IQR = Q3 − Q1. Đánh dấu khi x < Q1 − 1,5 × IQR hoặc x > Q3 + 1,5 × IQR. Giá trị đúng bằng cận không bị đánh dấu. Tính cận và so sánh trước khi làm tròn hiển thị.

In [3]:
dong_tong_ket = []
dong_nhat_ky = []
for stt, r in enumerate(dong_phan_vi, start=1):
    cot = r["TenCot"]
    iqr = r["Q3"] - r["Q1"]
    can_duoi = r["Q1"] - Decimal("1.5") * iqr
    can_tren = r["Q3"] + Decimal("1.5") * iqr
    so_duoi = so_tren = 0
    for _, ho_so in du_lieu.iterrows():
        x = ho_so[cot]
        if x is None:
            continue
        vi_tri = "Dưới cận" if x < can_duoi else "Trên cận" if x > can_tren else None
        if vi_tri is not None:
            so_duoi += int(vi_tri == "Dưới cận")
            so_tren += int(vi_tri == "Trên cận")
            dong_nhat_ky.append({"STT":stt, "TenCot":cot, "MaHoSo":ho_so["MaHoSo"],
                "GiaTri":x, "CanDuoi":can_duoi, "CanTren":can_tren, "ViTri":vi_tri})
    dong_tong_ket.append({"TenCot":cot, "IQR":iqr, "CanDuoi":can_duoi,
        "CanTren":can_tren, "SoDuoiCan":so_duoi, "SoTrenCan":so_tren,
        "TongNgoaiLe":so_duoi + so_tren})
ngoai_le = pd.DataFrame(dong_tong_ket)
nhat_ky = pd.DataFrame(dong_nhat_ky).sort_values(
    ["STT", "GiaTri", "MaHoSo"], ignore_index=True)
hien_thi_2_so(ngoai_le, ["IQR", "CanDuoi", "CanTren"])

,TenCot,IQR,CanDuoi,CanTren,SoDuoiCan,SoTrenCan,TongNgoaiLe
0,age,15.00,30.50,90.50,0,3,3
1,anaemia,1.00,-1.50,2.50,0,0,0
2,creatinine_phosphokinase,462.00,-573.00,1275.00,0,26,26
3,diabetes,1.00,-1.50,2.50,0,0,0
4,ejection_fraction,15.00,7.50,67.50,0,2,2
5,high_blood_pressure,1.00,-1.50,2.50,0,0,0
6,platelets,81500.00,95250.00,421250.00,8,15,23
7,serum_creatinine,0.40,0.40,2.00,0,34,34
8,serum_sodium,5.00,126.50,146.50,6,1,7
9,sex,1.00,-1.50,2.50,0,0,0


## 4. Ví dụ các ô bị đánh dấu
Hiển thị tối đa 3 ví dụ mỗi cột như SQL. Nhật ký vẫn lưu đầy đủ mọi ô bị đánh dấu. Một hồ sơ có thể có ngoại lệ ở nhiều cột; số ô ngoại lệ không phải số bệnh nhân.

In [4]:
vi_du = nhat_ky.groupby("STT", sort=False).head(3).drop(columns="STT").reset_index(drop=True)
hien_thi_2_so(vi_du, ["GiaTri", "CanDuoi", "CanTren"])

,TenCot,MaHoSo,GiaTri,CanDuoi,CanTren,ViTri
0,age,31,94.00,30.50,90.50,Trên cận
1,age,27,95.00,30.50,90.50,Trên cận
2,age,56,95.00,30.50,90.50,Trên cận
3,creatinine_phosphokinase,47,1380.00,-573.00,1275.00,Trên cận
4,creatinine_phosphokinase,177,1419.00,-573.00,1275.00,Trên cận
5,creatinine_phosphokinase,140,1548.00,-573.00,1275.00,Trên cận
6,ejection_fraction,218,70.00,7.50,67.50,Trên cận
7,ejection_fraction,65,80.00,7.50,67.50,Trên cận
8,platelets,278,25100.00,95250.00,421250.00,Dưới cận
9,platelets,16,47000.00,95250.00,421250.00,Dưới cận


**Nhận xét:** Có 95 ô bị đánh dấu ở 6 cột: age (3), creatinine_phosphokinase (26), ejection_fraction (2), platelets (23), serum_creatinine (34), serum_sodium (7). Đây là ngoại lệ thống kê, chưa kết luận là lỗi dữ liệu. Với các cột mã danh mục/nhị phân, IQR chỉ là khảo sát số học; phải kiểm tra miền mã riêng. DEATH_EVENT = 2 nằm trong cận IQR nhưng vẫn sai miền 0/1 và sẽ được xử lý tại phần không nhất quán.

## 5. Lưu các bảng khảo sát
Không sửa CSV nguồn và không tạo dữ liệu after ở bước này.

In [5]:
thu_muc_kq = thu_muc / "ket_qua/14_ngoai_le/01_khao_sat"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
five_number.to_csv(thu_muc_kq / "five_number_python.csv", index=False, encoding="utf-8-sig")
ngoai_le.to_csv(thu_muc_kq / "ngoai_le_python.csv", index=False, encoding="utf-8-sig")
nhat_ky.to_csv(thu_muc_kq / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu khảo sát; giữ nguyên dữ liệu đầu vào."]})

,TrangThai
0,Đã lưu khảo sát; giữ nguyên dữ liệu đầu vào.
